# 3 — Espaço, gravidade e segregação

No notebook 2 a geografia ficou de fora: tratamos a rede como um grafo abstrato. Agora ela entra —
e essa é a maior vantagem de ter trocado a pessoa pela antena.

Com a pessoa como nó, o mapa era só um pano de fundo: desenhávamos pontos espalhados dentro da
célula da antena para não sobrepor tudo, mas aquela posição era fictícia. Agora **cada célula do
mapa é literalmente um nó** e **cada linha desenhada é literalmente uma aresta**. O mapa deixou de
ser ilustração e virou a própria estrutura de dados.

Isso destrava três análises que antes não eram possíveis:

1. **Mapas temáticos por região** — insularidade, balanço emissor/receptor, uso per capita.
2. **Modelo de gravidade** — prever o fluxo a partir do tamanho das regiões e da distância.
3. **Homofilia socioeconômica entre territórios** — com um modelo nulo que faz sentido.

## 1. Preparação

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 40)

from src.utils import load_config
from src import antenna

CIDADE = "campinas"
config = load_config(CIDADE)
config["spatial"]["download_basemap"] = True   # False para rodar offline

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

# net traz nós, fluxos, grafo, dirigido e backbone — tudo construído no notebook 2
net = antenna.build(edges_antenna, antennas, config)
nodes, flows = net.nodes, net.flows

print(f"{net.n_antennas} regiões | {net.G.number_of_edges():,} fluxos | "
      f"{nodes['n_users'].sum():,} moradores")

## 2. As regiões no mapa

As antenas viram pontos georreferenciados e, a partir deles, **células de Voronoi**: cada ponto do
território é atribuído à antena mais próxima. É a aproximação padrão para a "área de influência" de
uma antena, e é o que nos permite pintar a cidade inteira.

As funções de construção estão em `src/pipeline/spatial.py` — o notebook as reaproveita em vez de
duplicar a reconstrução das células (que é uma rotina geométrica longa e sem valor didático aqui).

In [ ]:
from src.pipeline.spatial import _build_gdf, _build_voronoi, _add_basemap

gdf = _build_gdf(nodes)          # pontos em EPSG:3857 (métrico, casa com os tiles)
vor = _build_voronoi(gdf)        # células de Voronoi recortadas

# a extensão real é medida em lon/lat: o EPSG:3857 infla distâncias por 1/cos(latitude)
lon, lat = nodes["lon"], nodes["lat"]
print(f"{len(gdf)} antenas georreferenciadas")
print(f"extensão real: {antenna.haversine_km(lon.min(), lat.mean(), lon.max(), lat.mean()):.0f} km "
      f"leste-oeste, {antenna.haversine_km(lon.mean(), lat.min(), lon.mean(), lat.max()):.0f} km "
      f"norte-sul")

centro_campinas = (-47.0608, -22.9099)
d_centro = antenna.haversine_km(lon.to_numpy(), lat.to_numpy(), *centro_campinas)
print(f"antenas a mais de 15 km do centro: {(d_centro > 15).sum()} de {len(d_centro)} "
      f"(a mais distante, {d_centro.max():.0f} km)")

> ⚠️ **Um alerta sobre a abrangência dos dados.** Todas as 145 antenas estão marcadas com
> `residence_city == "Campinas"`, mas elas se espalham por cerca de **48 × 39 km**, e 46 delas
> estão a mais de 15 km do centro. No mapa aparecem pontos sobre Americana, Santa Bárbara d'Oeste,
> Sumaré, Hortolândia, Paulínia, Valinhos e Jaguariúna — municípios vizinhos.
>
> Ou o campo `residence_city` designa a **Região Metropolitana de Campinas**, ou as geometrias são
> anonimizadas de forma grosseira. Isso **não invalida nenhuma análise**, mas muda o enquadramento
> da apresentação: provavelmente estamos falando da RMC, não do município. Vale confirmar.

In [ ]:
def mapa(coluna, titulo, cmap, categorico=False, centro=None):
    """Pinta as células de Voronoi por um atributo das regiões."""
    fig, ax = plt.subplots(figsize=(11, 11))
    kw = dict(column=coluna, cmap=cmap, legend=True, edgecolor="black",
              linewidth=0.4, alpha=0.65, ax=ax)
    if categorico:
        kw["categorical"] = True
    elif centro is not None:
        v = vor[coluna].dropna()
        span = max(abs(v.min() - centro), abs(v.max() - centro))
        kw.update(vmin=centro - span, vmax=centro + span)
    vor.plot(**kw)
    gdf.plot(ax=ax, color="black", markersize=5)
    _add_basemap(ax, config)
    ax.set_axis_off()
    ax.set_title(titulo)
    plt.show()


mapa("residence_quintile_state", f"A geografia da renda — {config['city_name']}",
     "RdYlGn", categorico=True)

Este é o pano de fundo socioeconômico de tudo que vem depois. Note a distribuição bem desigual das
antenas por estrato:

In [ ]:
resumo_quintil = nodes.groupby("residence_quintile_state").agg(
    regioes=("antenna_id", "size"),
    moradores=("n_users", "sum"),
    chamadas=("calls_total", "sum"),
    insularidade_media=("insularity", "mean"),
).round(3)
resumo_quintil["%_do_volume"] = (100 * resumo_quintil["chamadas"]
                                 / resumo_quintil["chamadas"].sum()).round(1)
resumo_quintil

Em Campinas, **q5 tem 66 das 145 regiões e q2 apenas 7**. Isso é fundamental para interpretar a
homofilia mais adiante: se um estrato concentra 41% do volume da cidade, ele vai aparecer muito nas
conversas de todo mundo — inclusive nas dele mesmo — **sem que isso seja preferência nenhuma**. É
exatamente por isso que precisamos de um modelo nulo.

## 3. Insularidade: onde a vida acontece no bairro

Aqui a métrica que criamos ao mudar a unidade de análise ganha um mapa.

In [ ]:
mapa("insularity", f"Insularidade: fração do volume que não sai da região — {config['city_name']}",
     "magma_r")

Escuro = região que fala consigo mesma. Regiões muito insulares tendem a ser **autossuficientes**
(têm comércio, serviços e trabalho por perto) ou **isoladas** (mal conectadas ao resto da cidade) —
duas leituras opostas que só o contexto local distingue, e que valem uma conversa com quem conhece
a cidade.

Para a gestão pública, é o argumento mais direto do projeto: se 36% da comunicação é interna ao
bairro, **descentralizar serviços acompanha a vida real das pessoas**.

## 4. Balanço emissor/receptor: onde as pessoas ligam para

O `net_balance` é `(emitidas − recebidas) / total`. Positivo = a região liga mais do que recebe.

In [ ]:
mapa("net_balance", f"Emissoras (vermelho) × receptoras (azul) — {config['city_name']}",
     "coolwarm", centro=0.0)

Regiões **receptoras** (azuis) são candidatas naturais a concentrar emprego, comércio e serviços —
é para lá que a cidade liga. Regiões **emissoras** (vermelhas) tendem a ser majoritariamente
residenciais.

É uma pista sobre a geografia econômica da cidade obtida **sem nenhum dado econômico** — só com o
padrão de quem liga para quem.

In [ ]:
mapa("calls_per_user", f"Chamadas por morador — {config['city_name']}", "viridis")

Este mapa controla o tamanho da região: mostra **intensidade de uso**, não população. Sem essa
normalização, qualquer mapa de volume acabaria sendo apenas um mapa de densidade demográfica.

## 5. Os corredores da cidade

Agora as arestas no mapa. Primeiro todos os 5.817 fluxos, depois só o backbone construído no
notebook 2 — a comparação mostra por que o filtro é necessário.

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString


def mapa_fluxos(df, titulo, valor="q_calls", cor="crimson"):
    xy = gdf.set_index("antenna_id").geometry
    sel = df[df["a"].isin(xy.index) & df["b"].isin(xy.index)].copy()
    sel["geometry"] = [LineString([xy[a], xy[b]]) for a, b in zip(sel["a"], sel["b"])]
    sel = gpd.GeoDataFrame(sel, geometry="geometry", crs=gdf.crs)

    fig, ax = plt.subplots(figsize=(11, 11))
    sel.plot(ax=ax, color=cor, linewidth=0.2 + 4.0 * sel[valor] / sel[valor].max(), alpha=0.45)
    gdf.plot(ax=ax, color="black", markersize=12)
    _add_basemap(ax, config)
    ax.set_axis_off()
    ax.set_title(titulo)
    plt.show()


mapa_fluxos(flows, f"Todos os {len(flows):,} fluxos entre regiões — {config['city_name']}")

Uma mancha ilegível — é a densidade 0,557 aparecendo visualmente. Nenhuma leitura é possível daqui.

In [ ]:
backbone_flows = pd.DataFrame(
    [(u, v, d["q_calls"]) for u, v, d in net.backbone.edges(data=True)],
    columns=["a", "b", "q_calls"],
)
mapa_fluxos(backbone_flows,
            f"Backbone: os {len(backbone_flows)} corredores estruturantes — {config['city_name']}",
            cor="darkred")

Agora sim. Os mesmos dados, filtrados pelo teste de significância, revelam o **esqueleto da
cidade** — e é esta figura que vai para a apresentação, não a anterior.

## 6. As macro-regiões no mapa — o teste decisivo

No notebook 2 o Louvain dividiu a cidade em 5 macro-regiões usando **apenas os pesos dos fluxos**.
O algoritmo não recebeu nenhuma coordenada, nenhuma distância, nada geográfico.

Se essas macro-regiões saírem espalhadas pelo mapa, elas são só um agrupamento estatístico. Se
saírem **contíguas**, significa que a divisão funcional da cidade — quem conversa com quem —
coincide com a divisão territorial. Vamos ver.

In [ ]:
from networkx.algorithms.community import louvain_communities

comunidades = louvain_communities(net.G, weight="weight", seed=42)
region_of = {a: i for i, com in enumerate(comunidades) for a in com}
nodes["macro_region"] = nodes["antenna_id"].map(region_of)
vor["macro_region"] = vor["antenna_id"].map(region_of)

mapa("macro_region", f"Macro-regiões detectadas só pelos fluxos — {config['city_name']}",
     "tab10", categorico=True)

**Elas saem contíguas.** Blocos territoriais coerentes — noroeste, norte-centro, leste/centro, sul,
sudeste — sem que o algoritmo soubesse onde qualquer antena fica.

Esse é o resultado visual mais forte do projeto, e a leitura para o "prefeito" é direta: **existe
uma cidade real, desenhada pelos fluxos cotidianos, que pode não coincidir com a cidade
administrativa.** Sobrepor este mapa às divisões oficiais de distritos ou regiões administrativas
mostra onde o desenho institucional não acompanha a vida das pessoas.

## 7. O modelo de gravidade

O "decaimento com a distância" já aparecia na análise antiga, mas só como uma curva caindo. Com
regiões, dá para **modelar** e dizer *quanto* cai.

A hipótese de gravidade, emprestada da física e clássica em geografia urbana, é:

$$F_{ij} \approx C \cdot \frac{(n_i \, n_j)^a}{d_{ij}^{\,b}}$$

O fluxo entre duas regiões cresce com o produto das populações e cai com a distância. Tirando log
dos dois lados, vira uma regressão linear simples:

$$\log F_{ij} = \log C + a \log(n_i n_j) - b \log d_{ij}$$

Basta um mínimos quadrados. Os coeficientes têm leitura direta:
- **`a`** — quanto o tamanho puxa o fluxo (1 = proporcional ao produto das populações)
- **`b`** — quão rápido a distância mata a interação (1 = decaimento clássico, 1/d)
- **`R²`** — quanto do fluxo tamanho e distância conseguem explicar

In [ ]:
fit = antenna.fit_gravity_model(flows)

print(f"expoente de tamanho    a = {fit['size_exponent']:.3f}")
print(f"expoente de distância  b = {fit['distance_exponent']:.3f}")
print(f"R²                       = {fit['r2']:.3f}   ({fit['n_flows']:,} fluxos)")

**b = 1,09** — praticamente o decaimento clássico 1/d. É um resultado bonito: a interação entre
bairros de Campinas obedece à mesma lei que descreve fluxos de migração, comércio e transporte em
cidades do mundo inteiro.

**a = 0,56** — sublinear. Regiões grandes falam **menos** do que o proporcional ao seu tamanho:
dobrar a população de duas regiões não dobra o fluxo entre elas, aumenta cerca de 47%. Faz sentido —
há um limite de quantos laços uma pessoa mantém.

**R² = 0,27** — e aqui está a parte mais interessante. Tamanho e distância explicam só **27%** da
variação. Os outros 73% são história, trabalho, origem comum, transporte. Não são ruído: são o
sinal que o modelo *não* captura.

In [ ]:
df = fit["flows"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df["dist_km"], df["q_calls"], s=8, alpha=0.25, color="steelblue")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("distância entre regiões (km)")
axes[0].set_ylabel("chamadas no fluxo")
axes[0].set_title(f"Decaimento com a distância (b = {fit['distance_exponent']:.2f})")
axes[1].scatter(df["log_predicted"], df["log_observed"], s=8, alpha=0.25, color="darkorange")
lims = [df["log_predicted"].min(), df["log_predicted"].max()]
axes[1].plot(lims, lims, "--", color="black", lw=1)
axes[1].set_xlabel("log do fluxo previsto")
axes[1].set_ylabel("log do fluxo observado")
axes[1].set_title(f"Previsto vs. observado (R² = {fit['r2']:.2f})")
plt.tight_layout(); plt.show()

### Os resíduos: onde a geografia não explica

O resíduo é `log(observado) − log(previsto)`. Resíduo alto = duas regiões falam **muito mais** do
que o tamanho e a distância delas justificariam.

Esses pares são a parte mais acionável da análise: sugerem dependências de trabalho, deslocamento
pendular ou populações com origem em comum.

In [ ]:
top_residuos = df.nlargest(15, "residual")[
    ["a", "b", "q_calls", "dist_km", "n_pairs", "residual"]
].round(2)
print("Pares que mais superam a previsão do modelo:")
top_residuos

In [ ]:
mapa_fluxos(df.nlargest(150, "residual"),
            f"Ligações muito acima do previsto pela gravidade — {config['city_name']}",
            valor="residual", cor="darkviolet")

Compare com o mapa do backbone: lá apareciam os fluxos **grandes**, que em boa parte são apenas
regiões grandes e próximas. Aqui aparecem os fluxos **surpreendentes** — muitos deles ligando
pontos distantes. São hipóteses de investigação, não conclusões: cada linha longa pede uma
explicação local.

## 8. Homofilia socioeconômica

A pergunta: **as regiões falam preferencialmente com regiões do mesmo quintil?**

Duas decisões de método importam aqui:

1. **Ponderar por volume**, não contar arestas. Numa rede com densidade 0,56 quase toda região se
   liga a quase toda região; o que distingue é *quanto*.
2. **Comparar com um modelo nulo.** Como vimos na seção 2, q5 sozinho concentra 41% do volume —
   então q5↔q5 seria frequente mesmo sem preferência nenhuma. O nulo embaralha os rótulos de
   quintil **entre as regiões**, mantendo os fluxos no lugar, e mede quanto sobraria por acaso.

In [ ]:
q = nodes.set_index("antenna_id")["residence_quintile_state"]
qa = q.reindex(flows["a"]).to_numpy()
qb = q.reindex(flows["b"]).to_numpy()
w = flows["q_calls"].to_numpy(dtype=float)

observado = w[qa == qb].sum() / w.sum()

# modelo nulo: embaralha o quintil entre regiões, mantendo a estrutura dos fluxos
indice = pd.Index(nodes["antenna_id"])
ia, ib = indice.get_indexer(flows["a"]), indice.get_indexer(flows["b"])
codigos = q.reindex(nodes["antenna_id"]).to_numpy()
rng = np.random.default_rng(42)
nulo = np.mean([w[perm[ia] == perm[ib]].sum() / w.sum()
                for perm in (rng.permutation(codigos) for _ in range(200))])

print(f"volume entre regiões do MESMO quintil: {observado:.1%}")
print(f"esperado ao acaso:                     {nulo:.1%}")
print(f"razão:                                 {observado / nulo:.2f}x")

**1,12×.** Praticamente nada — bem longe do "1,9×" que a análise no nível das pessoas produzia.

Isso **não** é um erro de cálculo, e o notebook 4 é inteiramente dedicado a explicar por quê. Por
ora, guarde o número.

Mas a média global esconde comportamentos opostos entre os estratos. Vamos abrir.

In [ ]:
QUINTIS = ["q1", "q2", "q3", "q4", "q5"]

# matriz de mistura em "meias-arestas": cada fluxo soma peso nas duas direções
mix = pd.DataFrame(0.0, index=QUINTIS, columns=QUINTIS)
for origem, destino, valor in zip(qa, qb, w):
    if origem in mix.index and destino in mix.columns:
        mix.loc[origem, destino] += valor
        mix.loc[destino, origem] += valor

e = mix / mix.to_numpy().sum()       # e[i][j] = fração do volume ligando i a j
a_i = e.sum(axis=1)                  # fração do volume que toca o quintil i

# índice de auto-preferência: observado dentro do estrato ÷ esperado se fosse ao acaso
auto_preferencia = pd.Series({qi: e.loc[qi, qi] / (a_i[qi] ** 2) for qi in QUINTIS})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(e.div(a_i, axis=0), annot=True, fmt=".2f", cmap="rocket_r", ax=axes[0])
axes[0].set_title("Para onde vai o volume de cada quintil")
axes[0].set_xlabel("quintil do outro lado"); axes[0].set_ylabel("quintil de origem")
axes[1].bar(auto_preferencia.index, auto_preferencia.values,
            color=["crimson" if v > 1 else "steelblue" for v in auto_preferencia])
axes[1].axhline(1.0, color="black", ls="--", lw=1)
axes[1].set_xlabel("quintil (q1 = 20% mais pobres … q5 = 20% mais ricos)")
axes[1].set_ylabel("volume interno observado ÷ esperado")
axes[1].set_title("Índice de fechamento de cada estrato")
plt.tight_layout(); plt.show()

auto_preferencia.round(2).to_frame("auto-preferência")

**Aqui está o que sobrevive — e é mais interessante que a média.**

- **q5 = 1,23** — as regiões mais ricas concentram a comunicação em si mesmas 23% acima do esperado.
- **q1 = 0,69** e **q2 = 0,41** — as regiões mais pobres falam consigo mesmas **bem menos** do que o
  acaso previria; elas se dispersam por todos os estratos.

As duas pontas da cidade se comportam de forma **oposta**. E a leitura para a gestão pública é
direta: quem depende do resto da cidade para trabalhar se comunica com o resto da cidade; quem não
depende, se fecha.

> Repare também na linha `q1` do heatmap: o maior volume dela vai para **q4 e q5**, não para q1.
> Isso é o padrão de quem atravessa a cidade para trabalhar.

## Síntese

| | Campinas |
|---|---|
| Expoente de distância (gravidade) | **b = 1,09** (decaimento clássico 1/d) |
| Expoente de tamanho | a = 0,56 (sublinear) |
| R² do modelo | 0,27 — a maior parte do fluxo tem outras causas |
| Homofilia entre regiões | 35% observado vs. 31% ao acaso = **1,12×** |
| Auto-preferência q5 / q1 | **1,23** / **0,69** |
| Macro-regiões no mapa | **contíguas**, sem o algoritmo saber geografia |

**Próximo passo:** o notebook 4 fecha o projeto com robustez, rich-club e — o mais importante — a
explicação de por que a homofilia caiu de 1,9× para 1,12×.